## Compute Weekly Market Regime Features

In [0]:
from pyspark.sql.functions import (
    col, weekofyear, year, avg, stddev, count, sum as spark_sum,
    when, lit, round as spark_round, min as spark_min, max as spark_max,
    collect_list, array, concat_ws, date_trunc, first, last as spark_last,
    lag,
)
from pyspark.sql.window import Window

gold = spark.table("raghavan_trading_signals.gold.daily_features")

# Create a week identifier (year + week number)
gold_with_week = (
    gold
    .withColumn("week_start", date_trunc("week", col("trade_date")))
    .withColumn("year_week", concat_ws("-",
        year("trade_date").cast("string"),
        weekofyear("trade_date").cast("string")
    ))
)

## Aggregate Market-Wide Weekly Metrics
- These metrics describe "what the entire market looked like this week"

In [0]:
weekly_regimes = (
    gold_with_week
    .groupBy("week_start")
    .agg(
        # Overall market return (average across all stocks)
        spark_round(avg("daily_return"), 6).alias("avg_market_return"),

        # Market volatility (average volatility across stocks)
        spark_round(avg("volatility_20d"), 6).alias("avg_volatility"),

        # Dispersion: how different are stock returns from each other?
        spark_round(stddev("daily_return"), 6).alias("return_dispersion"),

        # Momentum: what fraction of stocks have positive returns?
        spark_round(
            spark_sum(when(col("daily_return") > 0, 1).otherwise(0)) / count("*"),
            4
        ).alias("pct_stocks_positive"),

        # RSI distribution: fraction of stocks in oversold (<30) territory
        spark_round(
            spark_sum(when(col("rsi_14") < 30, 1).otherwise(0)) / count("*"),
            4
        ).alias("pct_stocks_oversold"),

        # RSI distribution: fraction of stocks in overbought (>70) territory
        spark_round(
            spark_sum(when(col("rsi_14") > 70, 1).otherwise(0)) / count("*"),
            4
        ).alias("pct_stocks_overbought"),

        # Average RSI across the market
        spark_round(avg("rsi_14"), 4).alias("avg_rsi"),

        # MACD: fraction of stocks with bullish MACD (histogram > 0)
        spark_round(
            spark_sum(when(col("macd_histogram") > 0, 1).otherwise(0)) / count("*"),
            4
        ).alias("pct_macd_bullish"),

        # Bollinger Band position: average across stocks
        spark_round(avg("bb_position"), 4).alias("avg_bb_position"),

        # Bollinger Band width: market-wide volatility gauge
        spark_round(avg("bb_width"), 6).alias("avg_bb_width"),

        # Volume: is the market trading above or below average?
        spark_round(avg("volume_ratio"), 4).alias("avg_volume_ratio"),

        # VIX: fear gauge (take the last value of the week)
        spark_round(spark_last("vix", ignorenulls=True), 4).alias("weekly_vix"),

        # Yield curve slope
        spark_round(spark_last("yield_curve_slope", ignorenulls=True), 4).alias("weekly_yield_curve"),

        # Fed funds rate
        spark_round(spark_last("fed_funds_rate", ignorenulls=True), 4).alias("weekly_fed_rate"),

        # Number of trading days this week (for completeness checking)
        count("*").alias("stock_day_count"),
    )
    .orderBy("week_start")
)

print(f"Weekly regimes: {weekly_regimes.count()} weeks")

## Add Forward Returns
- For each week, what did the market do in the NEXT 1 and 2 weeks?
- This is what makes the Vector Search useful: "similar weeks led to X outcome."

In [0]:
regime_window = Window.orderBy("week_start")

weekly_regimes = (
    weekly_regimes
    .withColumn("next_1w_return",
        spark_round(lag("avg_market_return", -1).over(regime_window), 6)
    )
    .withColumn("next_2w_return",
        spark_round(
            lag("avg_market_return", -1).over(regime_window) +
            lag("avg_market_return", -2).over(regime_window),
            6
        )
    )
    .withColumn("next_1w_positive",
        when(lag("avg_market_return", -1).over(regime_window) > 0, lit(1)).otherwise(lit(0))
    )
)

Create the Regime Feature Vector
- Combine key metrics into a single array column.
- This is what Vector Search will index.

In [0]:
from pyspark.sql.functions import array

regime_vector_cols = [
    "avg_market_return",
    "avg_volatility",
    "return_dispersion",
    "pct_stocks_positive",
    "pct_stocks_oversold",
    "pct_stocks_overbought",
    "avg_rsi",
    "pct_macd_bullish",
    "avg_bb_position",
    "avg_bb_width",
    "avg_volume_ratio",
    "weekly_vix",
    "weekly_yield_curve",
    "weekly_fed_rate",
]

weekly_regimes = weekly_regimes.withColumn(
    "regime_vector",
    array(*[col(c).cast("float") for c in regime_vector_cols])
)

## Write to Gold

In [0]:
(
    weekly_regimes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("raghavan_trading_signals.gold.weekly_market_regimes")
)

print("Weekly market regimes table written!")

## Verification

In [0]:
regimes = spark.table("raghavan_trading_signals.gold.weekly_market_regimes")

print(f"Total weeks: {regimes.count()}")
print(f"Columns: {regimes.columns}")

# Show the most recent weeks
display(
    regimes
    .orderBy(col("week_start").desc())
    .select("week_start", "avg_market_return", "avg_volatility",
            "pct_stocks_oversold", "weekly_vix", "weekly_yield_curve",
            "next_1w_return", "next_1w_positive")
    .limit(10)
)

# Show the regime vector for one week
display(
    regimes
    .orderBy(col("week_start").desc())
    .select("week_start", "regime_vector")
    .limit(3)
)